# Phase 2 Benchmark: TPC-H Q1 + Q6

> Closeout benchmark for Phase 2 on active MXFrame lazy APIs.

This notebook benchmarks:
- MXFrame forced CPU
- MXFrame forced GPU (when MAX-compatible GPU is available)
- Polars baseline (optional)
- Pandas baseline

Queries covered:
- **Q1** grouped aggregates by `l_returnflag`, `l_linestatus`
- **Q6** filtered global revenue aggregate

In [1]:
#| hide
#| eval: false

import os
import platform
import sys
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

from mxframe import LazyFrame, Scan, col, lit
from mxframe.custom_ops import clear_cache

try:
    import polars as pl
    POLARS_AVAILABLE = True
except ImportError:
    POLARS_AVAILABLE = False
    print("Polars not installed: skipping Polars baseline.")

from max import driver as _driver

In [10]:
#| hide
#| eval: false

def _safe_gpu_count() -> int:
    try:
        return int(_driver.accelerator_count())
    except Exception:
        return 0

def report_runtime_versions() -> None:
    print("Runtime versions")
    print("================")
    print(f"numpy: {np.__version__}")
    print(f"pandas: {pd.__version__}")
    print(f"pyarrow: {pa.__version__}")
    if POLARS_AVAILABLE:
        print(f"polars: {pl.__version__}")
    else:
        print("polars: not installed")

def report_benchmark_context(dataset_rows: int, dataset_name: str = "lineitem") -> None:
    print("Benchmark context")
    print("=================")
    print(f"Python: {sys.version.split()[0]}")
    print(f"Platform: {platform.platform()}")
    print(f"CPU cores: {os.cpu_count()}")
    print(f"MAX accelerators: {_safe_gpu_count()}")
    print(f"Dataset: {dataset_name} rows={dataset_rows:,}")
    print(f"Polars available: {POLARS_AVAILABLE}")

def summarize_relative(rows, baselines=("Pandas", "Polars")):
    stats = {name: s for name, s in rows}
    for baseline in baselines:
        if baseline not in stats:
            continue
        print(f"\nRelative to {baseline} (min-ms ratio, <1 is faster):")
        for name, sample in rows:
            ratio = sample["min"] / stats[baseline]["min"]
            print(f"  {name:<24} {ratio:.2f}x")

def print_takeaway(rows, target="Pandas", label="MXFrame CPU (hot)", close=1.15, beat=0.95):
    stats = {name: s for name, s in rows}
    if target not in stats or label not in stats:
        return
    ratio = stats[label]["min"] / stats[target]["min"]
    if ratio <= beat:
        verdict = "beats"
    elif ratio <= close:
        verdict = "is close to"
    else:
        verdict = "lags"
    print(f"\nTakeaway: {label} {verdict} {target} ({ratio:.2f}x).")

## 1) Synthetic lineitem data

In [ ]:
#| hide
#| eval: false

CUTOFF_Q1 = 10471          # 1998-09-02
Q6_DATE_LO = 8766          # 1994-01-01
Q6_DATE_HI = 9131          # 1995-01-01 (exclusive)
Q6_DISC_LO = 0.05
Q6_DISC_HI = 0.07
Q6_QTY_HI = 24.0

def make_lineitem_arrow(n: int = 1_000_000, seed: int = 42) -> pa.Table:
    rng = np.random.default_rng(seed)

    rf_choices = rng.integers(0, 3, size=n)
    ls_choices = rng.integers(0, 2, size=n)

    rf = np.array(["A", "N", "R"], dtype=object)[rf_choices]
    ls = np.array(["F", "O"], dtype=object)[ls_choices]

    quantity = rng.uniform(1.0, 50.0, size=n).astype(np.float32)
    extendedprice = rng.uniform(900.0, 100_000.0, size=n).astype(np.float32)
    discount = rng.uniform(0.0, 0.10, size=n).astype(np.float32)
    tax = rng.uniform(0.0, 0.08, size=n).astype(np.float32)
    shipdate = rng.integers(8_000, 10_550, size=n).astype(np.int32)

    return pa.table({
        "l_returnflag": rf.tolist(),
        "l_linestatus": ls.tolist(),
        "l_quantity": quantity,
        "l_extendedprice": extendedprice,
        "l_discount": discount,
        "l_tax": tax,
        "l_shipdate": shipdate,
    })

N = 1_000_000
lineitem = make_lineitem_arrow(N)
print(f"Generated {lineitem.num_rows:,} rows with columns: {lineitem.column_names}")
report_benchmark_context(lineitem.num_rows)
report_runtime_versions()

Generated 1,000,000 rows with columns: ['l_returnflag', 'l_linestatus', 'l_quantity', 'l_extendedprice', 'l_discount', 'l_tax', 'l_shipdate']
Benchmark context
Python: 3.12.12
Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
CPU cores: 24
MAX accelerators: 2
Dataset: lineitem rows=1,000,000
Polars available: True


## 2) Query implementations (MXFrame, Polars, Pandas)

In [5]:
#| hide
#| eval: false

def _lf(tbl: pa.Table) -> LazyFrame:
    return LazyFrame(Scan(tbl))

def run_q1_mxframe(tbl: pa.Table, device: str = "cpu") -> pa.Table:
    lf = _lf(tbl)
    out = (
        lf
        .filter(col("l_shipdate") <= lit(CUTOFF_Q1))
        .groupby("l_returnflag", "l_linestatus")
        .agg(
            col("l_quantity").sum().alias("sum_qty"),
            col("l_extendedprice").sum().alias("sum_base_price"),
            (col("l_extendedprice") * (lit(1.0) - col("l_discount"))).sum().alias("sum_disc_price"),
            (col("l_extendedprice") * (lit(1.0) - col("l_discount")) * (lit(1.0) + col("l_tax"))).sum().alias("sum_charge"),
            col("l_quantity").mean().alias("avg_qty"),
            col("l_extendedprice").mean().alias("avg_price"),
            col("l_discount").mean().alias("avg_disc"),
            col("l_quantity").count().alias("count_order"),
        )
        .sort("l_returnflag", "l_linestatus")
        .compute(device=device)
    )
    return out

def _run_q6_fallback_arrow(tbl: pa.Table) -> pa.Table:
    mask = pc.and_kleene(pc.greater_equal(tbl["l_shipdate"], pa.scalar(Q6_DATE_LO, type=pa.int32())), pc.less(tbl["l_shipdate"], pa.scalar(Q6_DATE_HI, type=pa.int32())))
    mask = pc.and_kleene(mask, pc.greater_equal(tbl["l_discount"], pa.scalar(Q6_DISC_LO, type=pa.float32())))
    mask = pc.and_kleene(mask, pc.less_equal(tbl["l_discount"], pa.scalar(Q6_DISC_HI, type=pa.float32())))
    mask = pc.and_kleene(mask, pc.less(tbl["l_quantity"], pa.scalar(Q6_QTY_HI, type=pa.float32())))
    filt = tbl.filter(mask)
    revenue = pc.sum(pc.multiply(filt["l_extendedprice"], filt["l_discount"]))
    return pa.table({"revenue": [float(revenue.as_py() if revenue is not None else 0.0)]})

def run_q6_mxframe(tbl: pa.Table, device: str = "cpu") -> pa.Table:
    lf = _lf(tbl)
    try:
        return (
            lf
            .filter(
                (col("l_shipdate") >= lit(Q6_DATE_LO))
                & (col("l_shipdate") < lit(Q6_DATE_HI))
                & (col("l_discount") >= lit(Q6_DISC_LO))
                & (col("l_discount") <= lit(Q6_DISC_HI))
                & (col("l_quantity") < lit(Q6_QTY_HI))
            )
            .groupby()
            .agg((col("l_extendedprice") * col("l_discount")).sum().alias("revenue"))
            .compute(device=device)
        )
    except Exception as e:
        print(f"Q6 fallback to PyArrow path due to compute failure: {e}")
        return _run_q6_fallback_arrow(tbl)

def run_q1_pandas(tbl: pa.Table) -> pd.DataFrame:
    pdf = tbl.to_pandas()
    q1 = pdf[pdf["l_shipdate"] <= CUTOFF_Q1].copy()
    q1["disc_price"] = q1["l_extendedprice"] * (1.0 - q1["l_discount"])
    q1["charge"] = q1["disc_price"] * (1.0 + q1["l_tax"])

    out = (
        q1.groupby(["l_returnflag", "l_linestatus"], as_index=False)
        .agg(
            sum_qty=("l_quantity", "sum"),
            sum_base_price=("l_extendedprice", "sum"),
            sum_disc_price=("disc_price", "sum"),
            sum_charge=("charge", "sum"),
            avg_qty=("l_quantity", "mean"),
            avg_price=("l_extendedprice", "mean"),
            avg_disc=("l_discount", "mean"),
            count_order=("l_quantity", "count"),
        )
        .sort_values(["l_returnflag", "l_linestatus"])
        .reset_index(drop=True)
    )
    return out

def run_q6_pandas(tbl: pa.Table) -> float:
    pdf = tbl.to_pandas()
    mask = (
        (pdf["l_shipdate"] >= Q6_DATE_LO)
        & (pdf["l_shipdate"] < Q6_DATE_HI)
        & (pdf["l_discount"] >= Q6_DISC_LO)
        & (pdf["l_discount"] <= Q6_DISC_HI)
        & (pdf["l_quantity"] < Q6_QTY_HI)
    )
    return float((pdf.loc[mask, "l_extendedprice"] * pdf.loc[mask, "l_discount"]).sum())

def run_q1_polars(tbl: pa.Table):
    if not POLARS_AVAILABLE:
        return None
    pl_df = pl.from_arrow(tbl)
    return (
        pl_df
        .filter(pl.col("l_shipdate") <= CUTOFF_Q1)
        .with_columns([
            (pl.col("l_extendedprice") * (1.0 - pl.col("l_discount"))).alias("disc_price"),
            (pl.col("l_extendedprice") * (1.0 - pl.col("l_discount")) * (1.0 + pl.col("l_tax"))).alias("charge"),
        ])
        .group_by(["l_returnflag", "l_linestatus"])
        .agg([
            pl.col("l_quantity").sum().alias("sum_qty"),
            pl.col("l_extendedprice").sum().alias("sum_base_price"),
            pl.col("disc_price").sum().alias("sum_disc_price"),
            pl.col("charge").sum().alias("sum_charge"),
            pl.col("l_quantity").mean().alias("avg_qty"),
            pl.col("l_extendedprice").mean().alias("avg_price"),
            pl.col("l_discount").mean().alias("avg_disc"),
            pl.col("l_quantity").count().alias("count_order"),
        ])
        .sort(["l_returnflag", "l_linestatus"])
)

def run_q6_polars(tbl: pa.Table):
    if not POLARS_AVAILABLE:
        return None
    pl_df = pl.from_arrow(tbl)
    out = (
        pl_df
        .filter(
            (pl.col("l_shipdate") >= Q6_DATE_LO)
            & (pl.col("l_shipdate") < Q6_DATE_HI)
            & (pl.col("l_discount") >= Q6_DISC_LO)
            & (pl.col("l_discount") <= Q6_DISC_HI)
            & (pl.col("l_quantity") < Q6_QTY_HI)
        )
        .select((pl.col("l_extendedprice") * pl.col("l_discount")).sum().alias("revenue"))
    )
    return float(out["revenue"][0])

q1_cpu_preview = run_q1_mxframe(lineitem, device="cpu")
q6_cpu_preview = run_q6_mxframe(lineitem, device="cpu")
print("MXFrame Q1 CPU preview:")
print(q1_cpu_preview.to_pandas().to_string(index=False))
print(f"MXFrame Q6 CPU preview revenue: {float(q6_cpu_preview.column('revenue')[0].as_py()):.4f}")

MXFrame Q1 CPU preview:
l_returnflag l_linestatus    sum_qty  sum_base_price  sum_disc_price   sum_charge   avg_qty    avg_price  avg_disc  count_order
           A            F 4103107.75    8131914752.0    7725578752.0 8034785792.0 25.447050 50433.292969  0.049952     161241.0
           A            O 4139794.75    8178190848.0    7769572352.0 8080540672.0 25.557285 50488.582031  0.050003     161981.0
           N            F 4119782.00    8149893120.0    7741892096.0 8051399168.0 25.460300 50366.433594  0.049968     161812.0
           N            O 4113073.75    8155671552.0    7748141568.0 8058702336.0 25.469685 50502.953125  0.049982     161489.0
           R            F 4121115.75    8132211200.0    7725741568.0 8035167232.0 25.521063 50360.796875  0.050007     161479.0
           R            O 4101858.50    8113100800.0    7707712000.0 8016372224.0 25.494167 50425.128906  0.049970     160894.0
MXFrame Q6 CPU preview revenue: 40689616.0000


## 3) Baseline previews

In [6]:
#| hide
#| eval: false

q1_pd = run_q1_pandas(lineitem)
q6_pd = run_q6_pandas(lineitem)
print("Pandas Q1 preview:")
print(q1_pd.to_string(index=False))
print(f"Pandas Q6 revenue: {q6_pd:.4f}")

if POLARS_AVAILABLE:
    q1_pl = run_q1_polars(lineitem)
    q6_pl = run_q6_polars(lineitem)
    print("Polars Q1 preview:")
    print(q1_pl)
    print(f"Polars Q6 revenue: {q6_pl:.4f}")
else:
    q1_pl = None
    q6_pl = None

Pandas Q1 preview:
l_returnflag l_linestatus    sum_qty  sum_base_price  sum_disc_price   sum_charge   avg_qty    avg_price  avg_disc  count_order
           A            F 4103113.50    8131883520.0    7725592576.0 8034798592.0 25.447086 50433.101562  0.049952       161241
           A            O 4139794.00    8178198016.0    7769561600.0 8080580608.0 25.557281 50488.625000  0.050003       161981
           N            F 4119764.75    8149879296.0    7741915648.0 8051404800.0 25.460194 50366.347656  0.049968       161812
           N            O 4113074.75    8155675136.0    7748204032.0 8058711040.0 25.469690 50502.976562  0.049982       161489
           R            F 4121102.50    8132192256.0    7725764096.0 8035144704.0 25.520981 50360.679688  0.050007       161479
           R            O 4101842.50    8113126400.0    7707748864.0 8016333824.0 25.494068 50425.289062  0.049970       160894
Pandas Q6 revenue: 40689612.0000
Polars Q1 preview:
shape: (6, 10)
┌───────────┬─────

## 4) Correctness checks (Q1 + Q6)

In [7]:
#| hide
#| eval: false


def _assert_q1_close(mx_tbl: pa.Table, ref_df: pd.DataFrame, label: str, rtol: float = 1e-2, atol: float = 1e-4):
    mx_df = mx_tbl.to_pandas().sort_values(["l_returnflag", "l_linestatus"]).reset_index(drop=True)
    ref_df = ref_df.sort_values(["l_returnflag", "l_linestatus"]).reset_index(drop=True)

    assert len(mx_df) == len(ref_df), f"{label}: row-count mismatch {len(mx_df)} vs {len(ref_df)}"
    for key in ["l_returnflag", "l_linestatus"]:
        assert (mx_df[key].values == ref_df[key].values).all(), f"{label}: key mismatch in {key}"

    value_cols = [
        "sum_qty", "sum_base_price", "sum_disc_price", "sum_charge",
        "avg_qty", "avg_price", "avg_disc", "count_order",
    ]
    for colname in value_cols:
        mx_vals = mx_df[colname].astype(float).to_numpy()
        ref_vals = ref_df[colname].astype(float).to_numpy()
        if not np.allclose(mx_vals, ref_vals, rtol=rtol, atol=atol):
            raise AssertionError(f"{label}: mismatch in {colname}")

def _extract_q6_revenue(mx_tbl: pa.Table) -> float:
    return float(mx_tbl.column("revenue")[0].as_py())

_assert_q1_close(q1_cpu_preview, q1_pd, "Q1 MXFrame CPU vs Pandas")
q6_mx_cpu_val = _extract_q6_revenue(q6_cpu_preview)
assert np.isclose(q6_mx_cpu_val, q6_pd, rtol=1e-2, atol=1e-3), "Q6 MXFrame CPU vs Pandas mismatch"

if POLARS_AVAILABLE:
    _assert_q1_close(q1_cpu_preview, q1_pl.to_pandas(), "Q1 MXFrame CPU vs Polars")
    assert np.isclose(q6_mx_cpu_val, q6_pl, rtol=1e-2, atol=1e-3), "Q6 MXFrame CPU vs Polars mismatch"

print("Correctness checks passed for MXFrame CPU against selected baselines.")

Correctness checks passed for MXFrame CPU against selected baselines.


## 5) Benchmark setup

In [8]:
#| hide
#| eval: false


COLD_RUNS = 3   # runs with cache cleared each time (includes JIT compilation)
HOT_RUNS  = 5   # runs with cache warm (steady-state execution only)

def _time_once(fn):
    """Time a single invocation, return (ms, result)."""
    t0 = time.perf_counter()
    out = fn()
    return (time.perf_counter() - t0) * 1000.0, out

def _time_runs(fn, runs, warmup=0):
    """Time multiple runs after optional warmup, return list of ms."""
    for _ in range(warmup):
        fn()
    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        samples.append((time.perf_counter() - t0) * 1000.0)
    return samples

def _stats(samples):
    arr = np.array(samples)
    return {"min": arr.min(), "avg": arr.mean(), "median": np.median(arr)}

# ── GPU readiness ──
GPU_READY = False
if _driver.accelerator_count() > 0:
    try:
        _ = run_q6_mxframe(lineitem, device="gpu")
        GPU_READY = True
    except Exception as e:
        print(f"GPU not usable: {e}")

print(f"GPU ready: {GPU_READY}")
print(f"Cold runs: {COLD_RUNS} (cache cleared each run — includes ~4s JIT compilation)")
print(f"Hot runs:  {HOT_RUNS} (cache warm — steady-state execution only)")

GPU ready: True
Cold runs: 3 (cache cleared each run — includes ~4s JIT compilation)
Hot runs:  5 (cache warm — steady-state execution only)


## 6) Benchmark: Q1 + Q6 — Cold / Hot / Polars

- **Cold** = `clear_cache()` before every timed run (includes JIT compilation of MAX graph + Mojo kernels)
- **Hot** = graph already compiled, reuse cached model (this is the steady-state for repeated queries)

In [11]:
#| hide
#| eval: false

def _print_table(title, rows):
    """Print a simple results table. rows = [(label, stats_dict), ...]"""
    print(f"\n{title}")
    print("=" * len(title))
    print(f"  {'Engine':<24} {'Min (ms)':>10} {'Avg (ms)':>10} {'Median (ms)':>12}")
    print(f"  {'-'*24} {'-'*10} {'-'*10} {'-'*12}")
    for name, s in rows:
        print(f"  {name:<24} {s['min']:>10.1f} {s['avg']:>10.1f} {s['median']:>12.1f}")


# ─────────────────────────────────────────────────
#  Q1: Grouped aggregation
# ─────────────────────────────────────────────────
print("━" * 60)
print("  Q1 — Grouped Aggregation (filter + groupby + 8 aggs)")
print("━" * 60)

# Q1 COLD (cache cleared each run)
q1_cold = []
cold_samples = []
for _ in range(COLD_RUNS):
    clear_cache()
    ms, _ = _time_once(lambda: run_q1_mxframe(lineitem, device="cpu"))
    cold_samples.append(ms)
q1_cold.append(("MXFrame CPU (cold)", _stats(cold_samples)))

if GPU_READY:
    cold_samples = []
    for _ in range(COLD_RUNS):
        clear_cache()
        ms, _ = _time_once(lambda: run_q1_mxframe(lineitem, device="gpu"))
        cold_samples.append(ms)
    q1_cold.append(("MXFrame GPU (cold)", _stats(cold_samples)))

_print_table(f"Q1 COLD — includes JIT compilation ({N:,} rows)", q1_cold)

# Q1 HOT (cache warm, warmup=2 then measure)
q1_hot = []
_ = run_q1_mxframe(lineitem, device="cpu")
q1_hot.append(("MXFrame CPU (hot)", _stats(_time_runs(lambda: run_q1_mxframe(lineitem, device="cpu"), runs=HOT_RUNS))))

if GPU_READY:
    _ = run_q1_mxframe(lineitem, device="gpu")
    q1_hot.append(("MXFrame GPU (hot)", _stats(_time_runs(lambda: run_q1_mxframe(lineitem, device="gpu"), runs=HOT_RUNS))))

q1_hot.append(("Pandas", _stats(_time_runs(lambda: run_q1_pandas(lineitem), runs=HOT_RUNS, warmup=1))))
if POLARS_AVAILABLE:
    q1_hot.append(("Polars", _stats(_time_runs(lambda: run_q1_polars(lineitem), runs=HOT_RUNS, warmup=2))))

_print_table(f"Q1 HOT — steady-state, no compilation ({N:,} rows)", q1_hot)
summarize_relative(q1_hot)
print_takeaway(q1_hot, target="Pandas", label="MXFrame CPU (hot)")
if GPU_READY:
    print_takeaway(q1_hot, target="Pandas", label="MXFrame GPU (hot)")


# ─────────────────────────────────────────────────
#  Q6: Filtered global aggregate
# ─────────────────────────────────────────────────
print("\n")
print("━" * 60)
print("  Q6 — Filtered Global Aggregate (5 filters + sum)")
print("━" * 60)

# Q6 COLD
q6_cold = []
cold_samples = []
for _ in range(COLD_RUNS):
    clear_cache()
    ms, _ = _time_once(lambda: run_q6_mxframe(lineitem, device="cpu"))
    cold_samples.append(ms)
q6_cold.append(("MXFrame CPU (cold)", _stats(cold_samples)))

if GPU_READY:
    cold_samples = []
    for _ in range(COLD_RUNS):
        clear_cache()
        ms, _ = _time_once(lambda: run_q6_mxframe(lineitem, device="gpu"))
        cold_samples.append(ms)
    q6_cold.append(("MXFrame GPU (cold)", _stats(cold_samples)))

_print_table(f"Q6 COLD — includes JIT compilation ({N:,} rows)", q6_cold)

# Q6 HOT
q6_hot = []
_ = run_q6_mxframe(lineitem, device="cpu")
q6_hot.append(("MXFrame CPU (hot)", _stats(_time_runs(lambda: run_q6_mxframe(lineitem, device="cpu"), runs=HOT_RUNS))))

if GPU_READY:
    _ = run_q6_mxframe(lineitem, device="gpu")
    q6_hot.append(("MXFrame GPU (hot)", _stats(_time_runs(lambda: run_q6_mxframe(lineitem, device="gpu"), runs=HOT_RUNS))))

q6_hot.append(("Pandas", _stats(_time_runs(lambda: run_q6_pandas(lineitem), runs=HOT_RUNS, warmup=1))))
if POLARS_AVAILABLE:
    q6_hot.append(("Polars", _stats(_time_runs(lambda: run_q6_polars(lineitem), runs=HOT_RUNS, warmup=2))))

_print_table(f"Q6 HOT — steady-state, no compilation ({N:,} rows)", q6_hot)
summarize_relative(q6_hot)
print_takeaway(q6_hot, target="Pandas", label="MXFrame CPU (hot)")
if GPU_READY:
    print_takeaway(q6_hot, target="Pandas", label="MXFrame GPU (hot)")

# ── Revenue sanity check ──
q6_mx = _extract_q6_revenue(run_q6_mxframe(lineitem, device="cpu"))
print(f"\nQ6 revenue (MXFrame CPU): {q6_mx:.4f}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Q1 — Grouped Aggregation (filter + groupby + 8 aggs)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Q1 COLD — includes JIT compilation (1,000,000 rows)
  Engine                     Min (ms)   Avg (ms)  Median (ms)
  ------------------------ ---------- ---------- ------------
  MXFrame CPU (cold)           2999.8     3037.6       3021.5
  MXFrame GPU (cold)           6179.1     6260.0       6298.5

Q1 HOT — steady-state, no compilation (1,000,000 rows)
  Engine                     Min (ms)   Avg (ms)  Median (ms)
  ------------------------ ---------- ---------- ------------
  MXFrame CPU (hot)              14.0       49.7         14.6
  MXFrame GPU (hot)              32.6       33.1         33.3
  Pandas                         95.6       96.2         95.9
  Polars                         29.0       30.6         30.3

Relative to Pandas (min-ms ratio, <1 is faster):
  MXFrame CPU (hot)        0.15x
  MXFrame G